# Taxi Trip Exploratory Data Analysis

**Name(s)**: Drake Graham

**Website Link**: https://dgraham6.github.io/Taxi-EDA/

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import matplotlib.pyplot as plt

import requests
from tqdm import tqdm
pd.options.plotting.backend = 'plotly'
from lec_utils import *
import folium
from folium.plugins import MarkerCluster
import googlemaps
import time
import math
from sklearn.model_selection import train_test_split
%matplotlib inline
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer


## Step 1: Introduction

Here are some initial questions about the dataset:  
- On what days do taxi drivers generate the most revenue?  
- What is the relationship between trip characteristics, such as distance and duration, and revenue?  
- Are there significant differences in performance between taxi companies?  

After consideration, I decided to focus on a what I think is the most important: predicting how long a trip will last. This has practical applications in improving route efficiency, setting accurate expectations for customers, and optimizing fleet management for taxi companies.

## Step 2: Data Cleaning and Exploratory Data Analysis

In [11]:
#taxi = pd.read_csv('durations.csv')
#taxi = pd.read_csv('OpenDataDC_Taxi_2024/taxi_2024_10.csv')
#for i in range(1, 9):
   # taxi = pd.concat([taxi, pd.read_csv(f"OpenDataDC_Taxi_2024/taxi_2024_0{i}.csv")], axis=0)

## More Data

Before diving deep into the taxi log data we inhance our resources by retriveing hourly weather data uisng the Open-Meteo API and merge left on our data set. And for additional data to reference we can use OSRM API to get the predicted fastest route and route distnace for each of our trips

In [8]:
#import pandas as pd
#import requests
#from tqdm import tqdm
#osrm_url = "http://localhost:8080/route/v1/driving"


#def batch_coordinates(df, batch_size):
#    for i in range(0, len(df), batch_size):
#       yield df.iloc[i:i + batch_size]
#
#def format_coordinates(row):
#    return f"{row['ORIGIN_BLOCK_LONGITUDE']},{row['ORIGIN_BLOCK_LATITUDE']};{row['DESTINATION_BLOCK_LONG']},{row['DESTINATION_BLOCK_LAT']}"


#def get_route_duration_distance(coords):
#    try:
#        response = requests.get(f"{osrm_url}/{coords}?overview=false")
#        if response.status_code == 200:
#            data = response.json()
#            if 'routes' in data and len(data['routes']) > 0:
#                route = data['routes'][0]
#                return route.get('duration', None), route.get('distance', None)
#            else:
#                return None, None
#        else:
#            return None, None
#    except Exception as e:
#        print(f"Error: {e}")
#        return None, None


#durations, distances = [], []
#for batch in tqdm(batch_coordinates(taxi, batch_size=50), total=len(taxi) // 50 + 1):
#    for _, row in batch.iterrows():
#        coords = format_coordinates(row)
#        duration, distance = get_route_duration_distance(coords)
#        durations.append(duration)
#        distances.append(distance)
#        
#taxi['duration'] = durations
#taxi['distance'] = distances


In [10]:
import openmeteo_requests
import requests_cache
import pandas as pd
from retry_requests import retry

cache_session = requests_cache.CachedSession('.cache', expire_after=-1)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
openmeteo = openmeteo_requests.Client(session=retry_session)


url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": 38.9072, 
    "longitude": -77.0369,  
    "start_date": "2024-01-01", 
    "end_date": "2024-11-01",  
    "hourly": "snowfall,precipitation"  
}


responses = openmeteo.weather_api(url, params=params)
response = responses[0]  

hourly = response.Hourly()
hourly_snowfall = hourly.Variables(0).ValuesAsNumpy()  
hourly_precipitation = hourly.Variables(1).ValuesAsNumpy()


weather_data = {
    "time": pd.date_range(
        start=pd.to_datetime(hourly.Time(), unit="s", utc=True),
        end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),
        freq=pd.Timedelta(seconds=hourly.Interval()),
        inclusive="left"
    ),
    "snowfall": hourly_snowfall,
    "precipitation": hourly_precipitation
}
weather_df = pd.DataFrame(weather_data)

weather_df["time"] = pd.to_datetime(weather_df["time"], utc=True)
taxi["time"] = pd.to_datetime(taxi["Time"], utc=True)


taxi = pd.merge(taxi, weather_df, on="time", how="left")

Taking a look at the columns we can immidently rule certain columns as irevalent to the problem we want to attack. Things such as 'FAREAMOUNT', 'GRATUITYAMOUNT', and 'PAYMENTTYPE' are either completely unrelated or just a function of duraiton. We can salfey drop those columns to narrow our focus.

In [ ]:
taxi = taxi.drop(['TRIPTYPE', 'PROVIDERNAME','FAREAMOUNT', 
                  'GRATUITYAMOUNT', 'SURCHARGEAMOUNT', 'EXTRAFAREAMOUNT',
                  'TOLLAMOUNT', 'TOTALAMOUNT', 'PAYMENTTYPE'], axis=1)

There are a lot of NaN values, unfortunately if any of the coordinates of the Origin and Distance are not present there is no correct way to impute it, and that data is so crucial to our estimatation it leads us to just completly drop those columns.

In [ ]:
taxi = taxi.dropna(subset=['ORIGIN_BLOCK_LATITUDE', 'ORIGIN_BLOCK_LONGITUDE', 'DESTINATION_BLOCK_LAT', 'DESTINATION_BLOCK_LONG'])

In [ ]:
taxi = taxi.drop(taxi.loc[(taxi['DURATION'] < 1) | (taxi['DURATION'] > 3600)].index)

In [ ]:
taxi.groupby('precipitation')['DURATION'].median()

In [ ]:
taxi['Hour of Day'] = taxi['ORIGINDATETIME_TR'].dt.hour

grouped = taxi.groupby(['Day of the Week', 'Hour of Day']).agg(
    total_mileage=('MILEAGE', 'sum'),
    avg_duration=('DURATION', 'mean'),
    avg_temp=('temp', 'mean')
).reset_index()

pivot_table = grouped.pivot_table(
    index=['Day of the Week'],
    columns='Hour of Day',
    values=['total_mileage', 'avg_duration', 'avg_temp'],
    aggfunc='mean'
)

At first glance, abstract from the actual meanings of the values, we see many NaN values. In fact all of the 'PROVIDERNAME' values are NaN leading us to drop the column entirely, this is unfortunate as in other taxi trips datasets that may be useful data. Another row with common NaN values is 'AIRPORT', but becase there are non NaN values in this binary column we can treat the NaN values as an indicator that it is most likley false. We can further our confidence in the value of this column by using the origin coordinates to calculte if this trip did in fact come from the airport ourself.

Another problem that becomes evident is the existence of extreme duration, or distnace values in our data that can only be explained by input error.
While things such as a 3 minute drip down the block is feasible, as a taxi trip that lasts over three days, or a taxi trip that went over 50 miles can be safley erased from our data set do to its relativlely rare appreance.

In [ ]:
taxi.loc[taxi['DURATION'] > 172800, 'DURATION']
taxi['ORIGINCITY'] = taxi['ORIGINCITY'].apply(lambda x: x.lower() if isinstance(x, str) and 'WASHINGTON' in x.upper() else x)
taxi['DESTINATIONCITY'] = taxi['DESTINATIONCITY'].apply(lambda x: x.lower() if isinstance(x, str) and 'WASHINGTON' in x.upper() else x)

We can also parse our data time column to create day of the week features. We cna see the signficance in this graph

Here we can visulize these first 1000 pickup locations

In [ ]:
fig = px.scatter(taxi, x='snowfall', y='Duration(m)',
                         title='Heatmap of Weather Metric vs Trip Duration',
                         labels={'severerisk': 'Weather Metric', 'DURATION': 'Trip Duration (minutes)'},
                         )
fig.update_yaxes(title_text='Trip Duration(m)', range=[30, 50])
fig.show()

In [ ]:
taxi['Time']  = pd.to_datetime(taxi['ORIGINDATETIME_TR'], format='%m/%d/%Y %H:%M')
taxi['Duration(m)'] = taxi['DURATION'] / 60
taxi['Time'] = pd.to_datetime(taxi['Time'])
taxi['Day of the Week'] = taxi['Time'].dt.day_name()
taxi = pd.get_dummies(taxi, columns=['Day of the Week'])
week = taxi.groupby('Day of the Week', as_index=False)['Duration(m)'].count()
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
week['Day of the Week'] = pd.Categorical(week['Day of the Week'], categories=day_order, ordered=True)
week = week.sort_values('Day of the Week')
taxi['Day of the Week'] = week['Day of the Week']

In [ ]:
fig = px.bar(week, x='Day of the Week', y='Duration(m)', title='Trip Count by Day')
fig.update_yaxes(title_text='Count', range=[0, 200000])
fig.show()

In [ ]:
X = taxi.drop(taxi.loc[taxi['DURATION'] > 3600].index)

In [ ]:
taxi['Airport'] = taxi.apply(
    lambda row: True if pd.isna(row['AIRPORT']) and ((haversine(row['ORIGIN_BLOCK_LATITUDE'],row['ORIGIN_BLOCK_LONGITUDE', 38.9522, -77.4579]) < 20) or (haversine(row['DESTINATION_BLOCK_LAT'],row['DESTINATION_BLOCK_LONG']) < 10)) 
    else False,
    axis=1
)
taxi['ORIGINDATETIME_TR'] = pd.to_datetime(taxi['ORIGINDATETIME_TR'])

taxi['Hour of Day'] = taxi['ORIGINDATETIME_TR'].dt.hour

grouped = taxi.groupby(['Day of the Week', 'Hour of Day']).agg(
    total_mileage=('MILEAGE', 'sum'),
    avg_duration=('DURATION', 'mean'),
    avg_temp=('temp', 'mean')
).reset_index()

pivot_table = grouped.pivot_table(
    index=['Day of the Week'],
    columns='Hour of Day',
    values=['total_mileage', 'avg_duration', 'avg_temp'],
    aggfunc='mean'
)dont 
print(counts[['semester', 'Count']].head().to_markdown(index=False))
pivot_table.dropna().head()

In [ ]:
# TODO
taxi_clean = taxi.head(1000)

m = folium.Map(location=[38.9072, -77.0369], zoom_start=12) 

for _, row in taxi_clean.iterrows():
    folium.Marker(
        location=[row['ORIGIN_BLOCK_LATITUDE'], row['ORIGIN_BLOCK_LONGITUDE']]
    ).add_to(m)

m

In [ ]:
weather = pd.read_csv('Washington DC 2024-01-01 to 2024-11-20.csv')
taxi['Time']  = pd.to_datetime(taxi['ORIGINDATETIME_TR'], format='%m/%d/%Y %H:%M')
weather['Time'] = pd.to_datetime(weather['datetime'])
taxi = pd.merge(taxi, weather, on='Time', how = 'left')

We can also run OSRM API locally for free and gather estimates on the fastest route duration and distance usings its open source resources. This can give us values to comapre to and more data on the actual driving distnace of our trips

## Imputation
Unfortunately, for this dataset, if either the origin or destination coordinates are missing, it is impossible to reasonably estimate those values based on the available data. These two values are so critical that,  the rest of the data for that trip becomes unusable. Despite that, if our data does have coordinates we can use that to impute NaN Airport, and zip values.

## Step 3: Framing a Prediction Problem

As stated in the introduction, our goal is to use the data a taxi driver would feasibly have at the begning of a trip to predict the complete trip duration without implementing our own trip algroithms. We want to create a Regression model that predicts trip lenght using the like a gps software like Google maps would. Our target is to lower our Root Mean Squared Logarithmic Error. This metric is robust to outliers good with skewed data, and works well with our non negative values. Although it does have sensativity for small values, whcih is appropriate as small trips would be common in such a urban environemnt. 

## Step 4: Baseline Model

In our baseline model uses multiple quantive features like distance, precipation level, and hour of day but also qualtiive features like day of the weak, section of the city, and if its snowing or not.

In [ ]:
X = taxi.drop(['AIRPORT','DURATION', 'log_duration', 'duration', 'distance','Unnamed: 0','DESTINATIONDATETIME_TR', ], axis=1)
y = taxi['DURATION']
import numpy as np

def rmsle(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    epsilon = 1e-10
    y_true = np.maximum(y_true, epsilon)
    y_pred = np.maximum(y_pred, epsilon)
    

    log_diff = np.log1p(y_pred) - np.log1p(y_true)
    rmsle_value = np.sqrt(np.mean(log_diff ** 2))
    return rmsle_value


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=True)

In [ ]:
def baseline_model(X_train, y_train):
    rmsle_scorer = make_scorer(rmsle, greater_is_better=False)

    num_features = X_train.select_dtypes(include=['number']).columns
    cat_features = X_train.select_dtypes(include=['object']).columns

    num_preprocessor = Pipeline([
        ('imputer', SimpleImputer(strategy='mean')),
        ('scaler', StandardScaler())
    ])

    cat_preprocessor = Pipeline([
        ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
        ('encoder', OneHotEncoder(handle_unknown='ignore'))
    ])

    column_transformer = make_column_transformer(
        (num_preprocessor, num_features),
        (cat_preprocessor, cat_features),
        remainder='passthrough'
    )


    pipeline = Pipeline([
        ('preprocessor', column_transformer),
        ('regressor', LinearRegression())
    ])
    hyperparams = {}

    searcher = GridSearchCV(
        pipeline,
        param_grid=hyperparams,
        cv=5,
        scoring=rmsle_scorer
    )

    searcher.fit(X_train, y_train)

    return searcher
pipe_base = baseline_model(X_train, y_train)
pipe_base
# Once the above looks right, uncomment the expression below.
y_test_pred = pipe_base.predict(X_test)  # Ensure X_test matches y_test
rmsle_score = rmsle(y_test, y_test_pred)
rmsle_score

## Step 5: Final Model

In [ ]:
def compute_center(X):
    return X.assign(
        CenterLat=(X["ORIGIN_BLOCK_LATITUDE"] + X["DESTINATION_BLOCK_LAT"]) / 2,
        CenterLong=(X["ORIGIN_BLOCK_LONGITUDE"] + X["DESTINATION_BLOCK_LONG"]) / 2
    )
def compute_direction(X):
    return X.assign(
        Direction=X.apply(
            lambda row: (
                'NorthEast' if row['DESTINATION_BLOCK_LAT'] > row['ORIGIN_BLOCK_LATITUDE'] and row['DESTINATION_BLOCK_LONG'] > row['ORIGIN_BLOCK_LONGITUDE'] else
                'NorthWest' if row['DESTINATION_BLOCK_LAT'] > row['ORIGIN_BLOCK_LATITUDE'] and row['DESTINATION_BLOCK_LONG'] < row['ORIGIN_BLOCK_LONGITUDE'] else
                'SouthEast' if row['DESTINATION_BLOCK_LAT'] <= row['ORIGIN_BLOCK_LATITUDE'] and row['DESTINATION_BLOCK_LONG'] >= row['ORIGIN_BLOCK_LONGITUDE'] else
                'SouthWest'
            ),
            axis=1
        )
    )
def haversine(lat1, lon1, lat2, lon2):
    R = 3959
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    return R * c
    
def compute_distance(X):
    return X.assign(
        Distance=(haversine(X["ORIGIN_BLOCK_LATITUDE"], X["ORIGIN_BLOCK_LONGITUDE"],X["DESTINATION_BLOCK_LAT"], X["DESTINATION_BLOCK_LONG"]))
    )
    
def final_model(X_train, y_train):
    rmsle_scorer = make_scorer(rmsle, greater_is_better=False)

    add_center = FunctionTransformer(compute_center)
    add_direction = FunctionTransformer(compute_direction)
    add_distance = FunctionTransformer(compute_distance)

    num_features = X_train.select_dtypes(include=["number"]).columns
    cat_features = X_train.select_dtypes(include=["object"]).columns

    num_preprocessor = Pipeline([
        ("imputer", SimpleImputer(strategy="mean")),
        ("poly_features", PolynomialFeatures(degree=2, include_bias=False)),
        ("scaler", StandardScaler())
    ])
    cat_preprocessor = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ])

    column_transformer = ColumnTransformer(
        transformers=[
            ("num", num_preprocessor, num_features), 
            ("cat", cat_preprocessor, cat_features)   
        ],
        remainder="drop"
    )

    pipeline = Pipeline([
        ("add_distance", add_distance),
        ("add_center", add_center),
        ("add_direction", add_direction),
        ("preprocessor", column_transformer),
        ("regressor", LinearRegression())
    ])

    hyperparams = {
        "preprocessor__num__poly_features__degree": [1, 2, 3]  # Tune degree for polynomial features
    }

    searcher = GridSearchCV(
        pipeline,
        param_grid=hyperparams,
        cv=5,
        scoring=rmsle_scorer,
    )

    searcher.fit(X_train, y_train)
    return searcher
    
pipe_final = final_model(X_train, y_train)
y_test_pred = pipe_final.predict(X_test)
rmsle_score = rmsle(y_test, y_test_pred)
rmsle_score

## Sources: 

In [ ]:
X_train.columns